# Gold Skill Demand by Sector Mart

**Purpose**: Skill demand analysis aggregated by sector.

**Target Table**: `workspace.gold.gold_skill_demand_by_sector`

**Grain**: Date + sector + skill combinations

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 365)
- `force_full_refresh`: Boolean flag to force table recreation
- `top_n_skills`: Top N skills to keep per sector per day (default: 500)

**Key Metrics**:
- Job count requiring skill
- Company count
- Required vs preferred breakdown
- Demand rank within sector
- Penetration rate

**Data Sources**:
- `workspace.warehouse.bridge_job_skill`
- `workspace.warehouse.fact_job_postings`

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_skill_demand_by_sector_refresh_log`
- Captures: rows processed, unique dates/sectors/skills, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_skill_demand_by_sector"
metadata_table = "workspace.metadata.gold_skill_demand_by_sector_refresh_log"

# Parameters
lookback_days = 365  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table
top_n_skills = 500  # Keep top N skills per sector per day

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")
print(f"Top N Skills: {top_n_skills}")

In [0]:
%sql
-- Create table with all columns if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_skill_demand_by_sector_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  top_n_skills INT,
  rows_processed BIGINT,
  unique_dates INT,
  unique_sectors INT,
  unique_skills INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_skill_demand_by_sector refresh operations';

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_skill_demand_by_sector (
  trend_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  skill_sk BIGINT NOT NULL COMMENT 'FK to dim_skill',
  job_count BIGINT COMMENT 'Jobs requiring skill',
  company_count BIGINT COMMENT 'Distinct companies',
  required_count BIGINT COMMENT 'Jobs marking skill as required',
  preferred_count BIGINT COMMENT 'Jobs marking skill as preferred',
  demand_rank INT COMMENT 'Rank within sector',
  penetration_rate DECIMAL(10,4) COMMENT 'Skill usage as % of sector jobs',
  required_pct DECIMAL(10,4) COMMENT 'Required as % of total'
)
USING DELTA
COMMENT 'Skill demand analysis by sector'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
from datetime import datetime, timedelta

# Calculate cutoff date
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Building skill demand by sector (lookback: {lookback_days} days, cutoff: {cutoff_date})...")

# Populate data using INSERT OVERWRITE
insert_sql = f"""
INSERT OVERWRITE TABLE {target_table}

WITH base_skill_demand AS (
  SELECT
    f.posting_date_sk AS trend_date_sk,
    j.sector_sk,  -- Use sector_sk from dim_job for correct mapping
    bjs.skill_sk,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.job_sk END) AS job_count,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE THEN f.company_sk END) AS company_count,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE AND bjs.evidence_type = 'REQUIRED' THEN f.job_sk END) AS required_count,
    COUNT(DISTINCT CASE WHEN f.active_flag = TRUE AND bjs.evidence_type = 'PREFERRED' THEN f.job_sk END) AS preferred_count
  FROM workspace.warehouse.fact_job_postings f
  INNER JOIN workspace.warehouse.bridge_job_skill bjs 
    ON f.job_sk = bjs.job_sk AND bjs.is_current = TRUE
  INNER JOIN workspace.warehouse.dim_job j
    ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= {cutoff_date}
    AND j.sector_sk IS NOT NULL
    AND j.sector_sk != -1  -- Exclude unknown sector
    AND bjs.skill_sk IS NOT NULL
  GROUP BY f.posting_date_sk, j.sector_sk, bjs.skill_sk
),

sector_totals AS (
  SELECT
    trend_date_sk,
    sector_sk,
    SUM(job_count) AS sector_total_jobs
  FROM base_skill_demand
  GROUP BY trend_date_sk, sector_sk
),

with_rankings AS (
  SELECT
    bsd.*,
    DENSE_RANK() OVER (
      PARTITION BY bsd.trend_date_sk, bsd.sector_sk
      ORDER BY bsd.job_count DESC
    ) AS demand_rank,
    st.sector_total_jobs
  FROM base_skill_demand bsd
  INNER JOIN sector_totals st 
    ON bsd.trend_date_sk = st.trend_date_sk 
    AND bsd.sector_sk = st.sector_sk
)

SELECT
  trend_date_sk,
  sector_sk,
  skill_sk,
  job_count,
  company_count,
  required_count,
  preferred_count,
  demand_rank,
  CASE 
    WHEN sector_total_jobs > 0 
    THEN CAST(job_count AS DECIMAL(10,4)) / sector_total_jobs
    ELSE NULL 
  END AS penetration_rate,
  CASE 
    WHEN job_count > 0 
    THEN CAST(required_count AS DECIMAL(10,4)) / job_count
    ELSE NULL 
  END AS required_pct
FROM with_rankings
WHERE demand_rank <= {top_n_skills}
ORDER BY sector_sk, trend_date_sk DESC, demand_rank
"""

spark.sql(insert_sql)
print("✓ Data loaded")

In [0]:
import time

try:
    # Capture metrics
    end_time = time.time()
    processing_time = round(end_time - run_timestamp.timestamp(), 2)
    
    rows_processed = spark.table(target_table).count()
    
    metrics = spark.sql(f"""
        SELECT 
            COUNT(DISTINCT trend_date_sk) as unique_dates,
            COUNT(DISTINCT sector_sk) as unique_sectors,
            COUNT(DISTINCT skill_sk) as unique_skills
        FROM {target_table}
    """).collect()[0]
    
    unique_dates = metrics['unique_dates']
    unique_sectors = metrics['unique_sectors']
    unique_skills = metrics['unique_skills']
    
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed} rows")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Unique sectors: {unique_sectors}")
    print(f"✓ Unique skills: {unique_skills}")
    print(f"✓ Processing time: {processing_time}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    rows_processed = 0
    unique_dates = 0
    unique_sectors = 0
    unique_skills = 0
    processing_time = round(time.time() - run_timestamp.timestamp(), 2)
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Log to metadata table
    spark.sql(f"""
        INSERT INTO {metadata_table}
        VALUES (
            '{run_id}',
            timestamp'{run_timestamp.strftime("%Y-%m-%d %H:%M:%S")}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {top_n_skills},
            {rows_processed},
            {unique_dates},
            {unique_sectors},
            {unique_skills},
            {str(force_full_refresh).lower()},
            {processing_time},
            {f"'{error_message}'" if error_message else "NULL"}
        )
    """)

In [0]:
%sql
-- Validation: Summary Statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT trend_date_sk) AS unique_dates,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT skill_sk) AS unique_skills,
  MIN(trend_date_sk) AS earliest_date,
  MAX(trend_date_sk) AS latest_date,
  SUM(job_count) AS total_jobs,
  AVG(penetration_rate) AS avg_penetration_rate
FROM workspace.gold.gold_skill_demand_by_sector;

In [0]:
%sql
-- Top 10 skills per sector
SELECT
  ds.sector_name,
  dsk.skill_name,
  SUM(gsds.job_count) AS total_jobs,
  ROUND(AVG(gsds.penetration_rate), 4) AS avg_penetration,
  ROUND(AVG(gsds.required_pct), 4) AS avg_required_pct
FROM workspace.gold.gold_skill_demand_by_sector gsds
LEFT JOIN workspace.warehouse.dim_sector ds ON gsds.sector_sk = ds.sector_sk
LEFT JOIN workspace.warehouse.dim_skill dsk ON gsds.skill_sk = dsk.skill_sk
WHERE gsds.demand_rank <= 10
GROUP BY ds.sector_name, dsk.skill_name
ORDER BY ds.sector_name, total_jobs DESC
LIMIT 50;

In [0]:
%sql
-- Data quality checks
SELECT 
  'Null sector_sk' as check_name,
  COUNT(*) as issue_count
FROM workspace.gold.gold_skill_demand_by_sector
WHERE sector_sk IS NULL

UNION ALL

SELECT 
  'Null skill_sk' as check_name,
  COUNT(*) as issue_count
FROM workspace.gold.gold_skill_demand_by_sector
WHERE skill_sk IS NULL

UNION ALL

SELECT 
  'Negative job_count' as check_name,
  COUNT(*) as issue_count
FROM workspace.gold.gold_skill_demand_by_sector
WHERE job_count < 0

UNION ALL

SELECT 
  'Invalid penetration_rate' as check_name,
  COUNT(*) as issue_count
FROM workspace.gold.gold_skill_demand_by_sector
WHERE penetration_rate < 0 OR penetration_rate > 1;

In [0]:
%sql
-- View recent refresh logs
SELECT 
  run_id,
  run_timestamp,
  status,
  rows_processed,
  unique_dates,
  unique_sectors,
  unique_skills,
  lookback_days,
  top_n_skills,
  force_full_refresh,
  processing_time_seconds,
  error_message
FROM workspace.metadata.gold_skill_demand_by_sector_refresh_log
ORDER BY run_timestamp DESC
LIMIT 20;